# Judge Consistency and Significance Tests

Stage 3 asks whether the best methods selected in Stage 2 differ from their matching no-explanation baselines across multiple LLM judges.

This notebook is intentionally explicit:

1. Load one or more `data/consim_{judge}_v2.csv` files.
2. Restrict to best methods when an allow-list is available.
3. Canonicalize duplicated baseline rows by averaging them.
4. Pair method and baseline scores by seed.
5. Run paired Student t-tests.
6. Correct p-values across tested dataset cells.
7. Summarize judge consistency for the paper.

Reusable plotting lives in `utils/plot.py`; statistical logic stays in this notebook for auditability.


## 1. Parameters

Set `JUDGE_MODEL_FILES` to a list of score-file stems, or keep `None` to load all `data/consim_*_v2.csv` files.


In [ ]:
# None means load all data/consim_*_v2.csv files.
JUDGE_MODEL_FILES = None

# Stage 3 should usually use best prompts. While data/best_prompts/ is not
# populated yet, the notebook falls back to BEST_CONFIGS or all filtered rows.
ALLOW_LIST_PATH = None  # None -> data/best_prompts/allow_list.json
BEST_CONFIGS = None     # e.g. {"concepts": ["SemiNMF / topk"], "rationales": ["Qwen/Qwen3.5-2B"]}

# Optional filters. None means keep all values available after allow-listing.
DATASETS = None
CLASSES_SUBSETS = None
FAMILIES = None
PROMPT_TYPES = None
METHODS = None
SPECIFICATIONS = ["new_consim", "rationales", "attributions"]

# Statistical convention.
ALPHA = 0.05
MULTIPLE_COMPARISON_CORRECTION = "bh"  # "bh", "holm", or None
MIN_PAIRED_SEEDS = 2

# Export controls.
EXPORT_FIGURES = False
EXPORT_TABLES = False


## 2. Imports and Paths


In [ ]:
from pathlib import Path
import ast
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.plot import plot_ranked_score_bars
from utils.registries import ATTRIBUTION_METHOD_NAMES, CONCEPT_METHOD_NAMES

DATA_DIR = REPO_ROOT / "data"
BEST_PROMPTS_DIR = DATA_DIR / "best_prompts"
EXPORT_DIR = REPO_ROOT / "LaTeX-Simulatability-Shortcut" / "plots"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Score Files

Each loaded CSV gets a `judge` column from its filename stem.


In [ ]:
def judge_from_path(path: Path) -> str:
    name = path.name
    assert name.startswith("consim_") and name.endswith("_v2.csv")
    return name[len("consim_") : -len("_v2.csv")]

if JUDGE_MODEL_FILES is None:
    score_paths = sorted(DATA_DIR.glob("consim_*_v2.csv"))
else:
    score_paths = [DATA_DIR / f"consim_{stem}_v2.csv" for stem in JUDGE_MODEL_FILES]

missing = [path for path in score_paths if not path.exists()]
assert not missing, f"Missing score files: {missing}"
assert score_paths, "No v2 score files found."

frames = []
for path in score_paths:
    frame = pd.read_csv(path)
    frame["judge"] = judge_from_path(path)
    frames.append(frame)
    print(f"Loaded {len(frame):,} rows from {path.relative_to(REPO_ROOT)}")

df = pd.concat(frames, ignore_index=True)
print(f"Total rows: {len(df):,}")
print(f"judges: {sorted(df['judge'].unique())}")
print(f"specifications: {sorted(df['specification'].dropna().unique())}")
print(f"prompt_types: {sorted(df['prompt_type'].dropna().unique())}")
df.head()


## 4. Families, Baselines, and Filters

Baseline prompt rows are shared across families. We average duplicated baselines on their semantic key, but duplicated non-baseline rows are rejected.


In [ ]:
FAMILY_ORDER = ["concepts", "rationales", "attributions"]
FAMILY_COLORS = {
    "concepts": "#4c72b0",
    "rationales": "#55a868",
    "attributions": "#c44e52",
}

BASELINE_PROMPT_TYPES = {"B1", "B2", "AB1", "AB2"}
CONCEPT_PROMPT_TYPES = {"C1", "C2", "C3", "AC1", "AC2", "AC3"}
RATIONALE_PROMPT_TYPES = {"R1", "AR1"}
ATTRIBUTION_PROMPT_TYPES = {"A1", "AA1"}
BASELINE_FOR = {
    "C1": "B1", "C2": "B2", "C3": "B2",
    "AC1": "AB1", "AC2": "AB2", "AC3": "AB2",
    "R1": "B2", "AR1": "AB2",
    "A1": "B2", "AA1": "AB2",
}

CONCEPT_METHODS = set(CONCEPT_METHOD_NAMES) | set(CONCEPT_METHOD_NAMES.values())
ATTRIBUTION_METHODS = set(ATTRIBUTION_METHOD_NAMES) | {"Saliency", "IntegratedGradients", "Integrated Gradients", "SmoothGrad", "SquareGrad", "VarGrad", "GradientShap", "LIME", "KernelShap", "Occlusion", "Sobol"}
ROW_KEY_COLS = ["judge", "dataset", "model", "classes_subset", "seed", "method", "nb_concepts", "interpretation", "prompt_type", "specification"]
BASELINE_KEY_COLS = ["judge", "dataset", "model", "classes_subset", "seed", "prompt_type"]

def infer_family(row: pd.Series) -> str:
    prompt_type = str(row["prompt_type"])
    method = str(row["method"])
    if prompt_type in BASELINE_PROMPT_TYPES or method == "baseline":
        return "baseline"
    if prompt_type in CONCEPT_PROMPT_TYPES or method in CONCEPT_METHODS:
        return "concepts"
    if prompt_type in RATIONALE_PROMPT_TYPES:
        return "rationales"
    if prompt_type in ATTRIBUTION_PROMPT_TYPES or method in ATTRIBUTION_METHODS:
        return "attributions"
    return "unknown"

def family_config(row: pd.Series) -> str:
    method = str(row["method"])
    if row["family"] == "concepts":
        interpretation = row.get("interpretation")
        if pd.isna(interpretation) or str(interpretation) in {"None", "nan", "<NA>"}:
            return method
        return f"{method} / {interpretation}"
    return method

def apply_filter(frame: pd.DataFrame, column: str, values) -> pd.DataFrame:
    if values is None:
        return frame
    values = list(values)
    missing = set(values) - set(frame[column].dropna().unique())
    if missing:
        print(f"warning: {column} values not present before filtering: {sorted(missing)}")
    return frame[frame[column].isin(values)].copy()

def class_subset_len(value) -> float:
    try:
        return len(ast.literal_eval(str(value)))
    except (ValueError, SyntaxError, TypeError):
        return np.nan

def canonicalize_baselines_and_validate(frame: pd.DataFrame) -> pd.DataFrame:
    baseline_mask = frame["family"] == "baseline"
    baselines = frame[baseline_mask].copy()
    non_baselines = frame[~baseline_mask].copy()

    duplicate_non_baselines = (
        non_baselines.groupby(ROW_KEY_COLS, dropna=False).size().reset_index(name="n").query("n > 1")
    )
    if not duplicate_non_baselines.empty:
        display(duplicate_non_baselines.head(20))
        raise ValueError("Duplicated non-baseline score rows found. Inspect the displayed keys before continuing.")

    if baselines.empty:
        return non_baselines

    before = len(baselines)
    baseline_scores = baselines.groupby(BASELINE_KEY_COLS, dropna=False)["score"].mean().reset_index()
    collapsed = pd.DataFrame(columns=frame.columns)
    for column in baseline_scores.columns:
        collapsed[column] = baseline_scores[column]
    collapsed["method"] = "baseline"
    collapsed["nb_concepts"] = np.nan
    collapsed["interpretation"] = np.nan
    collapsed["specification"] = "baseline"
    collapsed["family"] = "baseline"
    collapsed["family_config"] = "baseline"
    after = len(collapsed)
    if before != after:
        print(f"Averaged duplicated baseline rows: {before:,} -> {after:,}")
    return pd.concat([non_baselines, collapsed], ignore_index=True)


## 5. Apply Allow-List and Filters

Stage 3 should normally use the allow-list exported from notebook 5. If no allow-list exists yet, this notebook warns and analyzes all filtered rows so the statistical code remains executable.


In [ ]:
df = df.copy()
df["family"] = df.apply(infer_family, axis=1)
df["family_config"] = df.apply(family_config, axis=1)

allow_path = Path(ALLOW_LIST_PATH) if ALLOW_LIST_PATH is not None else BEST_PROMPTS_DIR / "allow_list.json"
allow_list = None
if allow_path.exists():
    allow_list = json.loads(allow_path.read_text())
    print(f"Loaded allow-list from {allow_path.relative_to(REPO_ROOT)}")
else:
    print(f"No allow-list found at {allow_path.relative_to(REPO_ROOT)}; using BEST_CONFIGS or all filtered rows.")

df_f = df.copy()
df_f = apply_filter(df_f, "dataset", DATASETS)
df_f = apply_filter(df_f, "classes_subset", CLASSES_SUBSETS)
df_f = df_f[(df_f["dataset"] != "GE") | (df_f["classes_subset"].map(class_subset_len) == 3)].copy()
df_f = apply_filter(df_f, "family", FAMILIES)
df_f = apply_filter(df_f, "prompt_type", PROMPT_TYPES)
df_f = apply_filter(df_f, "method", METHODS)

# Apply specifications only to explanation rows. Baselines are canonicalized
# later and should remain available as shared comparators.
if SPECIFICATIONS is not None:
    spec_values = set(SPECIFICATIONS)
    df_f = df_f[(df_f["family"] == "baseline") | df_f["specification"].isin(spec_values)].copy()

if allow_list is not None:
    allowed_configs = []
    for family in FAMILY_ORDER:
        for entry in allow_list.get(family, []):
            allowed_configs.append((family, entry.get("family_config") or entry.get("method")))
    allowed_configs = set(allowed_configs)
    allowed_baselines = set(allow_list.get("baselines", ["B1", "B2", "AB1", "AB2"]))
    df_f = df_f[(df_f["family"] == "baseline") & df_f["prompt_type"].isin(allowed_baselines) | df_f.apply(lambda r: (r["family"], r["family_config"]) in allowed_configs, axis=1)].copy()
elif BEST_CONFIGS is not None:
    allowed_configs = {(family, config) for family, configs in BEST_CONFIGS.items() for config in (configs if isinstance(configs, list) else [configs])}
    df_f = df_f[(df_f["family"] == "baseline") | df_f.apply(lambda r: (r["family"], r["family_config"]) in allowed_configs, axis=1)].copy()
else:
    print("warning: no allow-list or BEST_CONFIGS; significance tests will cover all filtered non-baseline rows.")

df_f = df_f[df_f["family"].isin(set(FAMILY_ORDER) | {"baseline"})].copy()
df_f = canonicalize_baselines_and_validate(df_f)
print(f"Rows after filtering and baseline handling: {len(df_f):,}")
display(df_f.groupby(["judge", "family", "prompt_type"]).size().reset_index(name="n"))


## 6. Pair Methods with Matching Baselines

Each explanation prompt type has a matching no-explanation baseline. Scores are averaged to one value per seed before pairing.


In [ ]:
explanations = df_f[df_f["family"].isin(FAMILY_ORDER)].copy()
explanations = explanations[explanations["prompt_type"].isin(BASELINE_FOR)].copy()
explanations["baseline_prompt_type"] = explanations["prompt_type"].map(BASELINE_FOR)

baseline = df_f[df_f["family"] == "baseline"].copy()
baseline = baseline.rename(columns={"prompt_type": "baseline_prompt_type", "score": "baseline_score"})

METHOD_GROUP_COLS = ["judge", "dataset", "model", "classes_subset", "family", "family_config", "prompt_type", "baseline_prompt_type", "seed"]
BASELINE_GROUP_COLS = ["judge", "dataset", "model", "classes_subset", "baseline_prompt_type", "seed"]

method_seed = explanations.groupby(METHOD_GROUP_COLS, dropna=False)["score"].mean().reset_index()
baseline_seed = baseline.groupby(BASELINE_GROUP_COLS, dropna=False)["baseline_score"].mean().reset_index()
paired = method_seed.merge(
    baseline_seed,
    on=["judge", "dataset", "model", "classes_subset", "baseline_prompt_type", "seed"],
    how="inner",
)
paired["diff"] = paired["score"] - paired["baseline_score"]
print(f"Paired seed rows: {len(paired):,}")
paired.head()


## 7. Paired t-tests

The unit of pairing is the seed. Each test compares one method/configuration/prompt type against its matching baseline within one `(judge, dataset)` cell.


In [ ]:
TEST_GROUP_COLS = ["judge", "dataset", "family", "family_config", "prompt_type", "baseline_prompt_type"]

def run_one_test(group: pd.DataFrame) -> pd.Series:
    group = group.dropna(subset=["score", "baseline_score"])
    per_seed = group.groupby("seed", as_index=False)[["score", "baseline_score", "diff"]].mean().sort_values("seed")
    n = len(per_seed)
    mean_method = per_seed["score"].mean()
    mean_baseline = per_seed["baseline_score"].mean()
    mean_diff = per_seed["diff"].mean()
    std_diff = per_seed["diff"].std(ddof=1) if n > 1 else np.nan
    if n >= MIN_PAIRED_SEEDS and per_seed["diff"].nunique(dropna=True) > 1:
        statistic, p_value = ttest_rel(per_seed["score"], per_seed["baseline_score"])
    elif n >= MIN_PAIRED_SEEDS and per_seed["diff"].nunique(dropna=True) == 1:
        statistic = np.inf if mean_diff != 0 else 0.0
        p_value = 0.0 if mean_diff != 0 else 1.0
    else:
        statistic, p_value = np.nan, np.nan
    return pd.Series({
        "n_paired_seeds": n,
        "mean_method": mean_method,
        "mean_baseline": mean_baseline,
        "mean_diff": mean_diff,
        "std_diff": std_diff,
        "t_statistic": statistic,
        "p_value": p_value,
    })

tests = (
    paired.groupby(TEST_GROUP_COLS, dropna=False)
    .apply(run_one_test, include_groups=False)
    .reset_index()
) if not paired.empty else pd.DataFrame(columns=TEST_GROUP_COLS + ["n_paired_seeds", "mean_method", "mean_baseline", "mean_diff", "std_diff", "t_statistic", "p_value"])
tests.sort_values(["judge", "dataset", "family", "family_config", "prompt_type"]).head(30)


## 8. Multiple-Comparison Correction

Correction is applied within each `(judge, family, family_config, prompt_type)` family across dataset cells. This matches the paper-level claim unit: does this method beat baseline across task cells for a judge?


In [ ]:
def adjust_pvalues(p_values: pd.Series, method: str | None) -> pd.Series:
    p = p_values.astype(float)
    out = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna()
    m = len(valid)
    if m == 0 or method is None:
        return p

    order = valid.sort_values().index
    sorted_p = valid.loc[order].to_numpy()
    if method.lower() in {"bh", "fdr_bh"}:
        adjusted = sorted_p * m / np.arange(1, m + 1)
        adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    elif method.lower() == "holm":
        adjusted = sorted_p * (m - np.arange(m))
        adjusted = np.maximum.accumulate(adjusted)
    else:
        raise ValueError(f"Unknown correction method: {method}")
    adjusted = np.clip(adjusted, 0, 1)
    out.loc[order] = adjusted
    return out

tests = tests.copy()
if tests.empty:
    tests["p_corrected"] = []
else:
    tests["p_corrected"] = (
        tests.groupby(["judge", "family", "family_config", "prompt_type"], group_keys=False)["p_value"]
        .apply(lambda s: adjust_pvalues(s, MULTIPLE_COMPARISON_CORRECTION))
    )
tests["significant"] = tests["p_corrected"] < ALPHA
tests["beats_baseline"] = tests["mean_diff"] > 0
tests["significant_beats_baseline"] = tests["significant"] & tests["beats_baseline"]
tests.sort_values(["judge", "family", "family_config", "prompt_type", "dataset"]).head(30)


## 9. Judge Consistency Summary

This table is the main Stage 3 output: for each judge and method/config, how often does it beat baseline significantly?


In [ ]:
SUMMARY_GROUP_COLS = ["judge", "family", "family_config", "prompt_type"]
summary = (
    tests.groupby(SUMMARY_GROUP_COLS, dropna=False)
    .agg(
        n_cells=("p_value", "size"),
        mean_diff=("mean_diff", "mean"),
        median_diff=("mean_diff", "median"),
        n_beats_baseline=("beats_baseline", "sum"),
        n_significant=("significant", "sum"),
        n_significant_beats_baseline=("significant_beats_baseline", "sum"),
    )
    .reset_index()
    if not tests.empty else pd.DataFrame(columns=SUMMARY_GROUP_COLS + ["n_cells", "mean_diff", "median_diff", "n_beats_baseline", "n_significant", "n_significant_beats_baseline"])
)
if not summary.empty:
    summary["frac_significant_beats_baseline"] = summary["n_significant_beats_baseline"] / summary["n_cells"]
    summary["label"] = summary["family"] + "\n" + summary["family_config"].astype(str).str.slice(0, 28) + "\n" + summary["prompt_type"]
summary.sort_values(["judge", "frac_significant_beats_baseline", "mean_diff"], ascending=[True, False, False])


In [ ]:
for judge, sub in summary.groupby("judge", dropna=False):
    if sub.empty:
        continue
    ax = plot_ranked_score_bars(
        sub,
        label_col="label",
        value_col="mean_diff",
        err_col="_no_error_bar",
        color_col="family",
        color_map=FAMILY_COLORS,
        ylabel="Mean score difference vs baseline",
        ylim=(min(-0.05, float(sub["mean_diff"].min()) - 0.02), max(0.05, float(sub["mean_diff"].max()) + 0.02)),
        title=f"Judge consistency summary | {judge}",
        save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
        file_name=f"judge_consistency_{str(judge).replace('/', '-')}.pdf" if EXPORT_FIGURES else None,
    )
    # The grey text reports how many dataset cells significantly
    # beat baseline after correction.
    ordered = sub.sort_values("mean_diff", ascending=False).reset_index(drop=True)
    for i, row in ordered.iterrows():
        ax.text(i, row["mean_diff"], f"{int(row['n_significant_beats_baseline'])}/{int(row['n_cells'])}", ha="center", va="bottom", fontsize=8, color="dimgray")
    plt.show()


## 10. Paper Tables and Exports

The detailed table is useful for appendix/debugging. The summary table is the likely paper-facing artifact.


In [ ]:
paper_summary = summary.sort_values(["judge", "family", "family_config", "prompt_type"]).copy()
display(paper_summary)
display(tests.sort_values(["judge", "dataset", "family", "family_config", "prompt_type"]))

if EXPORT_TABLES:
    out_summary = DATA_DIR / "judge_consistency_summary.csv"
    out_tests = DATA_DIR / "judge_consistency_tests.csv"
    paper_summary.to_csv(out_summary, index=False)
    tests.to_csv(out_tests, index=False)
    print(f"Wrote {out_summary.relative_to(REPO_ROOT)}")
    print(f"Wrote {out_tests.relative_to(REPO_ROOT)}")


## 11. Notes for Redaction

After running the notebook, record:

- Which judges were included?
- Which best-method allow-list was used?
- For each family, how often did methods significantly beat baselines?
- Are results consistent across judges or judge-specific?
- Which table/plot should be inserted into `LaTeX-Simulatability-Shortcut/main.tex`?
